# 03 — Filter NYC Tweets

This notebook filters the raw Twitter CIKM 2010 dataset down to NYC-based users
and extracts a 20,000 tweet sample for sentiment analysis.

**Inputs:** `data/raw/twitter/twitter_cikm_2010/test_set_users.txt`, `test_set_tweets.txt`  
**Outputs:** `data/processed/nyc_users.csv`, `nyc_tweets.csv`, `nyc_sample_20k.csv`

In [8]:
import pandas as pd
import numpy as np

## Step 1: Load User Location Data

The raw user file is tab-separated with no header. Each row contains a user ID
and a location string in the format `UT: lat,lon`. Loaded with `latin-1` encoding
to handle special characters in the original dataset.

In [9]:
users_path = "../data/raw/twitter/twitter_cikm_2010/test_set_users.txt"

users = pd.read_csv(
    users_path,
    sep="\t",          # <-- FIXED
    header=None,
    names=["user_id", "location"],
    engine="python",
    encoding="latin-1"
)

users.head()


,user_id,location
0,22077441,"UT: 43.009815,-83.710408"
1,17145858,"UT: 38.856217,-90.056057"
2,33931268,"UT: 42.52548,-73.770351"
3,19521541,"UT: 40.735051,-74.228224"
4,26976263,"UT: 34.096153,-118.368732"


## Step 2: Parse Coordinates

Extract latitude and longitude from the location string using regex, convert to
numeric, and drop any rows where parsing failed.

In [10]:
# Extract lat and lon directly into two columns
users[["lat", "lon"]] = users["location"].str.extract(
    r"([-]?\d+\.\d+),([-]?\d+\.\d+)"
)

# Convert safely to numeric
users["lat"] = pd.to_numeric(users["lat"], errors="coerce")
users["lon"] = pd.to_numeric(users["lon"], errors="coerce")

# Drop invalid rows
users = users.dropna(subset=["lat", "lon"])

users.head()


,user_id,location,lat,lon
0,22077441,"UT: 43.009815,-83.710408",43.009815,-83.710408
1,17145858,"UT: 38.856217,-90.056057",38.856217,-90.056057
2,33931268,"UT: 42.52548,-73.770351",42.525480,-73.770351
3,19521541,"UT: 40.735051,-74.228224",40.735051,-74.228224
4,26976263,"UT: 34.096153,-118.368732",34.096153,-118.368732


## Step 3: Filter to NYC Bounding Box

Keep only users whose coordinates fall within NYC's geographic bounds:
- Latitude: 40.5 – 40.9
- Longitude: -74.3 – -73.7

This yields 1,010 NYC-based users out of 5,134 total.

In [11]:
print("Total clean users:", len(users))
users.describe()

Total clean users: 5134


,user_id,lat,lon
count,5.134000e+03,5134.000000,5134.000000
mean,3.429475e+07,37.550353,-87.464284
std,1.795916e+07,4.663917,16.231703
min,6.118000e+03,25.371254,-124.190764
25%,2.146989e+07,34.003968,-95.234417
50%,2.869947e+07,39.093412,-80.924899
75%,4.257031e+07,40.780975,-74.124791
max,1.039634e+08,48.826294,-66.661600


In [12]:
nyc_users = users[
    (users["lat"] >= 40.5) & (users["lat"] <= 40.9) &
    (users["lon"] >= -74.3) & (users["lon"] <= -73.7)
]

print("NYC users:", len(nyc_users))
nyc_users.head()

NYC users: 1010


,user_id,location,lat,lon
3,19521541,"UT: 40.735051,-74.228224",40.735051,-74.228224
10,33890321,"UT: 40.709022,-73.784337",40.709022,-73.784337
15,18710552,"UT: 40.647863,-73.922868",40.647863,-73.922868
18,60443903,"UT: 40.824584,-73.944458",40.824584,-73.944458
26,27025453,"UT: 40.694453,-73.863182",40.694453,-73.863182


## Step 4: Load Raw Tweet Data

The tweet file is also tab-separated with 5,108,333 total tweets. Each line
contains user ID, tweet ID, text, and timestamp. Text is extracted as everything
between the second and last tab to handle tweets that contain tab characters.

In [13]:
tweets_path = "../data/raw/twitter/twitter_cikm_2010/test_set_tweets.txt"

rows = []

with open(tweets_path, "r", encoding="latin-1") as f:
    for line in f:
        parts = line.strip().split("\t")
        
        if len(parts) >= 4:
            user_id = parts[0]
            tweet_id = parts[1]
            timestamp = parts[-1]
            text = "\t".join(parts[2:-1])  # everything between
            
            rows.append([user_id, tweet_id, text, timestamp])

tweets = pd.DataFrame(rows, columns=["user_id", "tweet_id", "text", "timestamp"])

tweets.head()

,user_id,tweet_id,text,timestamp
0,22077441,10538487904,Ok today I have to find something to wear for ...,2010-03-15 17:35:58
1,22077441,10536835844,I am glad I'm having this show but I can't wai...,2010-03-15 16:53:44
2,22077441,10536809086,Honestly I don't even know what's going on any...,2010-03-15 16:52:59
3,22077441,10534149786,@LovelyJ_Janelle hey sorry I'm sitting infront...,2010-03-15 15:42:07
4,22077441,10530203659,Sitting infront of this sewing machine ... I d...,2010-03-15 13:55:22


In [14]:
print("Total tweets loaded:", len(tweets))

Total tweets loaded: 5108333


## Step 5: Filter Tweets to NYC Users

Match tweets to NYC users by `user_id`. Both columns cast to string before
joining to avoid type mismatch. Results in 1,008,825 NYC tweets.

In [15]:
nyc_users["user_id"] = nyc_users["user_id"].astype(str)
tweets["user_id"] = tweets["user_id"].astype(str)

nyc_user_ids = set(nyc_users["user_id"])

nyc_tweets = tweets[tweets["user_id"].isin(nyc_user_ids)]

print("NYC Tweets:", len(nyc_tweets))


NYC Tweets: 1008825


## Step 6: Save Outputs

Save the full NYC user list, all NYC tweets, and a random 20,000-tweet sample
(`random_state=42` for reproducibility) to the processed data directory.
The 20k sample is used for sentiment analysis in notebook 04.

In [16]:
nyc_users.to_csv(
    "../data/processed/nyc_users.csv",
    index=False
)

In [17]:
nyc_tweets.to_csv(
    "../data/processed/nyc_tweets.csv",
    index=False
)


In [18]:
nyc_sample = nyc_tweets.sample(n=20000, random_state=42)

nyc_sample.to_csv(
    "../data/processed/nyc_sample_20k.csv",
    index=False
)
